Kafka es una plataforma distribuida de mensajería que permite que un sistema envíe datos y otro los reciba en tiempo real, sin que ambos tengan que estar conectados directamente.

**Conceptos**

- **Producer:** El que envía el dato (ej. un microservicio, un sensor).

- **Consumer:** El que lee el dato (ej. Spark Structured Streaming).

- **Broker:** El servidor de Kafka. Un "Cluster" es un conjunto de Brokers.

- **Topic:** La "categoría" o "tabla" donde se guardan los mensajes.

- **Offset:** Un número único que indica la posición de un mensaje dentro de una partición. Es vital para saber qué datos ya leímos.

![](https://media.geeksforgeeks.org/wp-content/uploads/20251112175818469025/apache_kafka_architecture.webp)

### Estructura de un Topic

Kafka no guarda todo en una sola bolsa. Divide los Topics en Particiones.

- **Paralelismo:** Las particiones permiten que Spark lea datos en paralelo (un Worker de Spark puede leer la Partición 1 mientras otro lee la Partición 2).

- **Orden:** Kafka solo garantiza el orden de los mensajes dentro de una misma partición.

![](https://miro.medium.com/0*6FVLBloofcJp_RsF.png)

In [0]:
df = (spark.readStream
  .format("kafka")
  .option("kafka.bootstrap.servers", "<server:ip>")
  .option("subscribe", "<topic>")
  .option("startingOffsets", "latest")
  .load()
)

In [0]:
df = spark.readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "host1:9092,host2:9092") \
  .option("subscribe", "ventas_real_time") \
  .option("startingOffsets", "latest") \
  .load()

In [0]:
df = (spark
  .read
  .format("kafka")
  .option("kafka.bootstrap.servers", "<server:ip>")
  .option("subscribe", "<topic>")
  .option("startingOffsets", "earliest")
  .option("endingOffsets", "latest")
  .load()
)

- `startingOffsets`: * `earliest`: Lee desde el principio del topic.

- `latest`: Lee solo lo que llegue a partir de ahora.

### **Schema**

| Column        | Type   |
|---------------|--------|
| `key`           | binary |
| `value`         | binary |
| `topic`         | string |
| `partition`     | int    |
| `offset`        | long   |
| `timestamp`     | long   |
| `timestampType` | int    |

**Tip de examen:** Como el `value` es binario, casi siempre debes convertirlo a String y luego procesar el JSON:
`df.selectExpr("CAST(value AS STRING)")`

### Configuration

| Option           | Value                                             | Description                             |
|------------------|---------------------------------------------------|-----------------------------------------|
| `subscribe`        | A comma-separated list of topics.                 | The topic list to subscribe to.         |
| `subscribePattern` | Java regex string.                                | The pattern used to subscribe to topic(s). |
| `assign`           | JSON string `{"topicA":[0,1],"topic":[2,4]}.`      | Specific topicPartitions to consume.    |

#### `failOnDataLoss`

- Si es `true` (por defecto), el stream fallará si Kafka borra datos que Spark aún no ha leído (porque expiró el tiempo de retención).

- Si es `false`, Spark ignorará los datos perdidos y seguirá procesando.

#### `maxOffsetsPerTrigger`

- Esto controla la velocidad. Le dice a Spark: "En cada micro-batch, no leas más de X mensajes".

- Es vital para evitar que tu cluster de Databricks se sature si hay un pico de datos repentino en Kafka.

In [0]:
.option("subscribePattern", "ventas_*")

### Lakeflow Declarative Pipelines

**Example:** Write to a streaming table from multiple Kafka topics

The following examples creates a streaming table named kafka_target and writes to that streaming table from two Kafka topics:

In [0]:
from pyspark import pipelines as dp

dp.create_streaming_table("kafka_target")

# Kafka stream from multiple topics
@dp.append_flow(target = "kafka_target")
def topic1():
  return (
    spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "host1:port1,...")
      .option("subscribe", "topic1")
      .load()
  )

@dp.append_flow(target = "kafka_target")
def topic2():
  return (
    spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "host1:port1,...")
      .option("subscribe", "topic2")
      .load()
  )

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE kafka_target;

CREATE FLOW
  topic1
AS INSERT INTO
  kafka_target BY NAME
SELECT * FROM
  read_kafka(bootstrapServers => 'host1:port1,...', subscribe => 'topic1');

CREATE FLOW
  topic2
AS INSERT INTO
  kafka_target BY NAME
SELECT * FROM
  read_kafka(bootstrapServers => 'host1:port1,...', subscribe => 'topic2');